In [1]:
import pandas as pd

def process_education(file_path, year):
    df = pd.read_csv(file_path, skiprows=1)

    df = df.rename(columns={
        "Geography": "GEO_ID",
        "Geographic Area Name": "NAME"
    })

    def get_col(text):
        matches = [c for c in df.columns if text in c]
        if not matches:
            raise KeyError(f"Missing: {text}")
        return matches[0]

    # 1. Did not graduate high school
    df["Did not graduate high school"] = df[[c for c in df.columns if any(x in c for x in [
        "No schooling", "Nursery", "Kindergarten",
        "1st grade","2nd grade","3rd grade","4th grade",
        "5th grade","6th grade","7th grade","8th grade",
        "9th grade","10th grade","11th grade",
        "12th grade, no diploma"
    ])]].apply(pd.to_numeric, errors="coerce").sum(axis=1)

    # 2. Graduated high school
    df["Graduated high school"] = (
        pd.to_numeric(df[get_col("Regular high school diploma")], errors="coerce") +
        pd.to_numeric(df[get_col("GED or alternative credential")], errors="coerce")
    )

    # 3. Attended college or technical school
    df["Attended college or technical school"] = df[[c for c in df.columns if any(x in c for x in [
        "Some college, less than 1 year",
        "Some college, 1 or more years, no degree",
        "Associate"
    ])]].apply(pd.to_numeric, errors="coerce").sum(axis=1)

    # 4. Graduated college or technical school
    df["Graduated college or technical school"] = df[[c for c in df.columns if any(x in c for x in [
        "Bachelor",
        "Master",
        "Professional school",
        "Doctorate"
    ])]].apply(pd.to_numeric, errors="coerce").sum(axis=1)

    # Convert to long format
    df_long = df.melt(
        id_vars=["GEO_ID", "NAME"],
        value_vars=[
            "Did not graduate high school",
            "Graduated high school",
            "Attended college or technical school",
            "Graduated college or technical school"
        ],
        var_name="education",
        value_name="population"
    )

    df_long["year"] = year

    df_long["prop"] = (
        df_long["population"] /
        df_long.groupby("GEO_ID")["population"].transform("sum")
    )

    return df_long

In [9]:
edu_2021 = process_education(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT1Y2021.B15003-Data.csv", 2021)
edu_2022 = process_education(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT1Y2022.B15003-Data.csv", 2022)
edu_2023 = process_education(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT1Y2023.B15003-Data.csv", 2023)
edu_2024 = process_education(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT1Y2024.B15003-Data.csv", 2024)

In [13]:
edu_2020 = process_education(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2020.B15003-Data.csv", 2020)

In [15]:
edu_all = pd.concat(
    [edu_2020, edu_2021, edu_2022, edu_2023, edu_2024],
    ignore_index=True
)

In [17]:
edu_all.to_csv("education_all_2020_2024.csv", index=False)

In [19]:
edu_all.groupby(["GEO_ID","year"])["prop"].sum().unique()

array([1., 1., 0.])

In [23]:
edu_all

,GEO_ID,NAME,education,population,year,prop
0,0500000US01001,"Autauga County, Alabama",Did not graduate high school,5813.0,2020,0.137348
1,0500000US01003,"Baldwin County, Alabama",Did not graduate high school,18689.0,2020,0.111708
2,0500000US01005,"Barbour County, Alabama",Did not graduate high school,5757.0,2020,0.284324
3,0500000US01007,"Bibb County, Alabama",Did not graduate high school,4464.0,2020,0.235443
4,0500000US01009,"Blount County, Alabama",Did not graduate high school,8830.0,2020,0.198896
...,...,...,...,...,...,...
26495,0500000US72113,"Ponce Municipio, Puerto Rico",Graduated college or technical school,29795.0,2024,0.261952
26496,0500000US72127,"San Juan Municipio, Puerto Rico",Graduated college or technical school,113479.0,2024,0.395473
26497,0500000US72135,"Toa Alta Municipio, Puerto Rico",Graduated college or technical school,23340.0,2024,0.364916
26498,0500000US72137,"Toa Baja Municipio, Puerto Rico",Graduated college or technical school,20655.0,2024,0.307943
